# Modèle français + kirundi (base : Pythia-14m)

Ce notebook enchaîne **tout** le projet, dans l'ordre :

| Étape | Ce qui se passe | Durée (GPU T4) |
|---|---|---|
| 0 | Préparer Colab : GPU, Google Drive, code | 2 min |
| 1 | Générer toutes les données (conjugaison, accords, Bible, cours...) | 10 à 20 min |
| 2 | Créer le tokenizer français + kirundi | 5 min |
| 3 | **Phase A** : bases de la langue sur le petit modèle (14M) | 1 à 2 h |
| 4 | **Agrandir** le modèle : 14M -> 122M (`grow.py`) | 1 min |
| 5 | **Phase B** : le grand modèle apprend tout | 6 à 10 h |
| 6 | **Phase D** : apprendre à répondre (questions / réponses, traduction) | 1 à 2 h |
| 7 | **Tests** : notes par catégorie + vitesse | 10 min |
| 8 | Essayer le modèle soi-même, le sauvegarder | |

**Si Colab coupe** (ça arrive après quelques heures) : relance les cellules de l'étape 0, puis la cellule de la phase en cours. `train.py` **reprend tout seul** au dernier point sauvegardé sur Google Drive. Les étapes déjà finies sont sautées automatiquement.

**Avant de commencer**, mets dans ton Google Drive, dossier `MyDrive/sources/` :
- `bible_kirundi.db`
- `bible_francais.db`

**Suivi à distance** : chaque étape écrit son journal dans `MyDrive/modele/journaux/` (données, phases A/B/D, notes des tests). Claude les lit dans ton Drive pour décider des corrections.

## Étape 0 : préparer Colab

In [ ]:
# 0.1  Vérifier le GPU (menu : Exécution > Modifier le type d'exécution > GPU T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 0.2  Brancher Google Drive : TOUT est sauvegardé là (données, modèles)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 0.3  Récupérer le code depuis GitHub
#      Ton dépôt : NiBienvenu/modele_train.
#      Dépôt privé : mets un token dans les Secrets de Colab (icône clé) sous le nom GH_TOKEN.
DEPOT = "https://github.com/NiBienvenu/modele_train.git"

import os
if not os.path.exists("/content/modele_train"):
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        DEPOT = DEPOT.replace("https://", f"https://{token}@")
    except Exception:
        pass  # dépôt public
    !git clone -q $DEPOT /content/modele_train
else:
    !cd /content/modele_train && git pull -q
%cd /content/modele_train
!pip install -q -r requirements.txt

In [ ]:
# 0.4  Les dossiers utilisés partout (sur Google Drive)
import os
RACINE  = "/content/drive/MyDrive"
SOURCES = f"{RACINE}/sources"          # bible_kirundi.db, bible_francais.db
D       = f"{RACINE}/donnees"          # données générées
QR      = f"{D}/qr"                    # questions / réponses (phase D)
M       = f"{RACINE}/modele"           # modèles
for d in (D, QR, M, f"{M}/journaux"):
    os.makedirs(d, exist_ok=True)
for f in ("bible_kirundi.db", "bible_francais.db"):
    print(("OK      " if os.path.exists(f"{SOURCES}/{f}") else "MANQUANT"), f"{SOURCES}/{f}")

def fait(chemin):
    """True si une étape est déjà finie (pour ne pas la refaire après une coupure)."""
    return os.path.exists(chemin)

## Étape 1 : générer les données

Chaque script écrit des fichiers `.txt` (phase A et B) et `.jsonl` (questions / réponses, phase D).
Les éléments réservés aux **tests** (10 % des phrases traduites, 5 % des verbes) sont **exclus** automatiquement (voir `separation.py`).

In [ ]:
# 1.1  Français : conjugaison de 7 015 verbes, 12 temps
if not fait(f"{D}/conjugaison_fr"):
    !python gen_conjugaison_fr.py --sortie $D/conjugaison_fr --sft $QR/conjugaison_fr.jsonl

In [ ]:
# 1.2  Kirundi : livre FSI (domaine public, téléchargé automatiquement)
if not fait(f"{D}/fsi_rn/fsi_rn.txt"):
    !python extraire_fsi.py --sortie $D/fsi_rn

In [ ]:
# 1.3  Bible kirundi + français (bases SQLite), passages alignés, part de 25 %
if not fait(f"{D}/bible/bible_bilingue_25.txt"):
    !python extraire_bible_db.py --rn $SOURCES/bible_kirundi.db --fr $SOURCES/bible_francais.db \
        --sortie $D/bible --sft $QR/traduction_bible.jsonl

In [ ]:
# 1.4  Lexique français <-> kirundi (ta liste de verbes)
#      On travaille sur des COPIES des listes de verbes : les fichiers du dépôt ne changent pas.
if not fait(f"{D}/lexique_rn/lexique_rn.txt"):
    !tail -n +2 $D/bible/mots_bible_rn.tsv | cut -f1 > /tmp/mots.tsv
    !cut -f1 $D/fsi_rn/mots_frequents.tsv >> /tmp/mots.tsv
    !cp data/verbes_rn.tsv /tmp/verbes_copie.tsv
    !python importer_dictionnaire.py --mots /tmp/mots.tsv --verbes /tmp/verbes_copie.tsv \
        --a_completer /tmp/a_completer.tsv --sortie $D/lexique_rn --sft $QR/lexique_rn.jsonl

In [ ]:
# 1.5  Cours de Cox : phrases traduites par une experte (+ tests mis de côté)
if not fait(f"{D}/cours/cours_bilingue.txt"):
    !python combiner_cours.py --resultats data/cours_resultats --sortie $D/cours --sft $QR/cours.jsonl

In [ ]:
# 1.6  Kirundi : conjugaison (3 478 verbes, 14 temps), accords (classes), pronoms objets, subordonnées
if not fait(f"{D}/conjugaison_rn"):
    !python gen_conjugaison_rn.py --sortie $D/conjugaison_rn --sft $QR/conjugaison_rn.jsonl
if not fait(f"{D}/grammaire_rn/accords_rn_001.txt"):
    !python gen_accords_rn.py --sortie $D/grammaire_rn --sft $QR/accords_rn.jsonl
if not fait(f"{D}/grammaire_rn/syntaxe_rn_001.txt"):
    !python gen_syntaxe_rn.py --sortie $D/grammaire_rn --sft $QR/syntaxe_rn.jsonl

In [ ]:
# 1.7  Le jeu de tests (568 questions) et le bilan des données
!python construire_tests.py 2>&1 | tee $M/journaux/donnees.txt
!du -sh $D/* | sort -h | tee -a $M/journaux/donnees.txt
!wc -l $QR/*.jsonl

## Étape 2 : le tokenizer

Créé **une seule fois**, sur **toutes** les données. Ensuite, il ne doit plus jamais changer
(sinon le modèle ne reconnaît plus ses propres numéros).

In [ ]:
if not fait(f"{M}/tokenizer_fr_rn"):
    !python train_tokenizer.py --source "$D/**/*.txt=1" --taille_vocab 24000 --sortie $M/tokenizer_fr_rn

## Étape 3 : phase A (petit modèle, 14M)

Le modèle apprend les morceaux de mots et les règles simples. Surveille deux chiffres :
- la **loss** doit descendre ;
- les **phrases d'exemple** affichées à chaque évaluation doivent devenir du vrai français / kirundi.

In [ ]:
if not fait(f"{M}/phaseA/finale"):
    !python -u train.py --config configs/phaseA.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a $M/journaux/phaseA.txt

## Étape 4 : agrandir le modèle (14M -> 122M)

Le savoir de la phase A est **gardé** : chaque tête d'attention est copiée à sa place, chaque couche est dupliquée.
Pour une autre taille, change `--hidden` / `--layers` / `--heads` (par exemple 768 / 16 / 12 = 150M, maximum conseillé 1024 / 12 / 16 = 200M).

In [ ]:
if not fait(f"{M}/grand_init"):
    !python grow.py --source $M/phaseA/finale --sortie $M/grand_init --hidden 768 --layers 12 --heads 12

## Étape 5 : phase B (grand modèle)

La plus longue. **Si Colab coupe, relance l'étape 0 puis cette cellule** : l'entraînement reprend où il s'était arrêté.
Si la mémoire du GPU déborde (*CUDA out of memory*), baisse `taille_lot` et augmente `accumulation` dans `configs/phaseB.yaml` (par exemple 8 et 16).

In [ ]:
if not fait(f"{M}/phaseB/finale"):
    !python -u train.py --config configs/phaseB.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a $M/journaux/phaseB.txt

## Étape 6 : phase D (apprendre à répondre)

In [ ]:
if not fait(f"{M}/phaseD/finale"):
    !python -u train.py --config configs/phaseD.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a $M/journaux/phaseD.txt

## Étape 7 : les tests

Pour chaque catégorie : **exact** (% de réponses parfaites) et **chrF** (ressemblance, la mesure habituelle des traductions, de 0 à 100).
Les catégories marquées **"NON (vraie preuve)"** n'ont jamais été vues à l'entraînement : ce sont elles qui montrent si le modèle a **compris**.

Tu peux aussi tester un modèle intermédiaire (`phaseB/finale`) pour comparer.

In [ ]:
!python -u evaluer.py --modele $M/phaseD/finale --montrer 3 --sortie $M/journaux/resultats_tests.jsonl 2>&1 | tee $M/journaux/notes_tests.txt

In [ ]:
# Voir les erreurs d'une catégorie en détail
!python evaluer.py --modele $M/phaseD/finale --categorie traduction_rn_fr --montrer 15 --sortie $M/journaux/resultats_traduction.jsonl

## Étape 8 : essayer le modèle et le sauvegarder

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

chemin = f"{M}/phaseD/finale"
tok = AutoTokenizer.from_pretrained(chemin)
model = AutoModelForCausalLM.from_pretrained(chemin).to("cuda").eval()

def demander(question, max_tokens=120):
    ids = tok(f"<|user|> {question} <|assistant|>", return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_tokens, do_sample=False,
                             eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for q in ["Traduis en français : Ndagenda ku kazi uyu musi.",
          "Traduis en kirundi : Les enfants qui étudient savent lire.",
          "Conjugue le verbe kwandika au futur (après aujourd'hui).",
          "Réponds en kirundi avec un pronom : Urabona igitabo?",
          "Comment dit-on « remercier » en kirundi ?"]:
    print("Q :", q)
    print("R :", demander(q), "\n")

In [ ]:
# Pose tes propres questions
print(demander("Traduis en français : Umwana arakora cane."))

In [ ]:
# Sauvegarde : un fichier .zip du modèle final (pour ton projet, Docker, etc.)
!cd $M/phaseD && zip -qr /content/modele_fr_rn.zip finale && ls -lh /content/modele_fr_rn.zip
!cp /content/modele_fr_rn.zip $M/
print("Copié dans Google Drive :", f"{M}/modele_fr_rn.zip")

# Option : le mettre sur Hugging Face (privé), décommente et mets ton nom :
# from huggingface_hub import login; login()
# model.push_to_hub("NiBienvenu/modele-fr-rn", private=True); tok.push_to_hub("NiBienvenu/modele-fr-rn", private=True)